In [0]:
%run ../config/config

In [0]:
import pyspark.sql.functions as F
from delta.tables import DeltaTable

spark.conf.set("spark.databricks.execution.timeout", "18000")

BRONZE_TABLE_PATH = f"{CATALOG}.{BRONZE_SCHEMA}.{BRONZE_TABLE}"
SILVER_TABLE_PATH = f"{CATALOG}.{SILVER_SCHEMA}.{SILVER_TABLE}"

In [0]:

df = spark.table(BRONZE_TABLE_PATH)

latest_date = (
    df
    .agg(F.max("ingested_at"))
    .first()[0]
)

print(latest_date)

df = df.filter(
    F.col("ingested_at") == latest_date
)

df = df.select(*COLUMNS_TO_KEEP)

#Cleaning data

In [0]:
#cleaning code
from pyspark.sql.window import Window

window = Window.partitionBy('code').orderBy(F.desc('last_modified_datetime'))

df = df.withColumn('flag', F.row_number() \
                   .over(window)) \
                   .filter("flag = 1") \
                   .drop('flag')

In [0]:
#cleaning url
df = df.withColumn(
    'url',
    F.when(
        ~F.col('url').contains('http') | F.col('url').isNull(),
        "N/A"
    ).otherwise(F.trim(F.col('url')))
)

In [0]:
#cleaning created_datetime, last_modified_datetime, last_updated_datetime and last_image_datetime
cols = [
    "created_datetime",
    "last_modified_datetime",
    "last_updated_datetime",
    "last_image_datetime"
]

for c in cols:
    df = df.withColumn(
        c,
        F.to_timestamp(c)
    )

In [0]:
#cleaning product_name, packaging, packaging_text, brands, manufacturing_places, purchase_places, ingredients_text, allergens, countries
cols = [
    'product_name',
    'packaging',
    'packaging_text',
    'brands',
    'manufacturing_places',
    'purchase_places',
    'ingredients_text',
    'allergens',
    'countries'
]

for c in cols:
    df = df.withColumn(
        c,
        F.regexp_replace(c, r'\b[a-zA-Z]{2}:', "")
    )
    df = df.withColumn(
        c,
        F.regexp_replace(c, r'^(0?[1-9]|1[0-2])\/\d{4}$', "")
    )

cols_to_capitalize_first_letter = [
    'product_name',
    'packaging',
    'packaging_text',
    'brands',
    'ingredients_text',
    'allergens',
]

for c in cols_to_capitalize_first_letter:
    df = df.withColumn(
        c,
        F.trim(
            F.concat(
                F.upper(F.substring(c, 1, 1)),
                F.lower(F.substring(c, 2, 1000))
            )
        )
    )

cols_to_initcap = [
    'manufacturing_places',
    'purchase_places',
    'countries'
]

for c in cols_to_initcap:
    df = df.withColumn(
        c,
        F.initcap(F.col(c))
    )

df = df.withColumn(
    'brands',
    F.when(
        F.col('brands').rlike(r'[!#"%&\^\*\(\)\-\+;:\'\`\$\~\.\[\]/<>@]') |
        F.col('brands').rlike(r'[0-9]$'),
        'N/A'
    ).otherwise(F.col('brands'))
)

country_mapper = F.create_map(
    [F.lit(x) for x in sum(COUNTRY_MAP.items(), ())]
)

country_cols = ['countries', 'purchase_places', 'manufacturing_places']
for col in country_cols:
    df = df.withColumn(
        col,
        F.when(
            F.lower(F.col(col)).isin(list(COUNTRY_MAP.keys())),
            country_mapper[F.col(col)]
        ).otherwise(F.col(col))
    )

In [0]:
#cleaning serving_quantity
df = df.withColumn(
    'serving_quantity',
    F.when(
        (F.col('serving_quantity').try_cast('float') < 0) | (F.col('serving_quantity').isNull()),
        0
    ).otherwise(F.col('serving_quantity').try_cast('float'))
)
df = df.withColumn(
    'serving_quantity',
    F.round(F.col('serving_quantity'), 2)
)

In [0]:
#cleaning nova_group
df = df.withColumn(
    'nova_group',
    F.when(
        ~F.col('nova_group').isin('1', '2', '3', '4') | (F.col('nova_group').isNull()),
        0
    ).otherwise(F.col("nova_group").try_cast('int'))
)

In [0]:
#cleaning pnns_groups_2
df = df.withColumn(
    'pnns_groups_2',
    F.when(
        ~F.col('pnns_groups_2').isin(PNN_CATEGORIES) | F.col('pnns_groups_2').isNull(),
        'N/A'
    ).otherwise(F.initcap(F.col('pnns_groups_2')))
)

In [0]:
#cleaning environmental_score_score and _grade
df = df.withColumn(
    'environmental_score_score',
    F.when(
        (F.col('environmental_score_score').try_cast('int') < 0) | 
        (F.col('environmental_score_score').try_cast('int') > 100) | 
        (F.col('environmental_score_score').isNull()),
        0
    ).otherwise(F.col('environmental_score_score').try_cast('int'))
)
df = df.withColumn(
    'environmental_score_grade',
    F.when(
        ~(F.col('environmental_score_grade').isin('a', 'b', 'c', 'd', 'e', 'f')) |
        (F.col("environmental_score_grade").isNull()),
        'N/A'
    ).otherwise(F.upper(F.col('environmental_score_grade')))
)

In [0]:
#cleaning product_quantity
df = df.withColumn(
  'product_quantity',
  F.when(
    (F.col('product_quantity').try_cast('float') < 0) | 
    F.col('product_quantity').isNull(),
    0
  ).otherwise(F.round(F.col('product_quantity').try_cast('float'), 2))
)

In [0]:
#cleaning owner
df = df.withColumn(
    'owner',
    F.regexp_replace('owner', r'org-', '')
)
df = df.withColumn(
    'owner',
    F.regexp_replace('owner', r'-', ' ')
)
df = df.withColumn(
    'owner',
    F.initcap(F.col('owner'))
)

In [0]:
#cleaning unique_scans_n
df = df.withColumn(
    'unique_scans_n',
    F.when(
    (F.col('unique_scans_n').try_cast('int') < 0) | 
    F.col('unique_scans_n').isNull(),
    0
  ).otherwise(F.col('unique_scans_n').try_cast('int'))
)

In [0]:
#cleaning completeness
df = df.withColumn(
    'completeness',
    F.col("completeness").try_cast('float') * 100
)
df = df.withColumn(
    'completeness',
    F.round(F.col('completeness'), 2)
)
df = df.withColumn(
    'completeness',
    F.concat(F.col('completeness').cast('string'), F.lit('%'))
)

In [0]:
#cleaning image_url columns
df = df.withColumn(
    'image_url',
    F.when(
        ~F.col('image_url').contains('http') | F.col('image_url').isNull(),
        "N/A"
    ).otherwise(F.trim(F.col('image_url')))
)
df = df.withColumn(
    'image_ingredients_url',
    F.when(
        ~F.col('image_ingredients_url').contains('http') | 
        F.col('image_ingredients_url').isNull(),
        "N/A"
    ).otherwise(F.trim(F.col('image_ingredients_url')))
)
df = df.withColumn(
    'image_nutrition_url',
    F.when(
        ~F.col('image_nutrition_url').contains('http') | 
        F.col('image_nutrition_url').isNull(),
        "N/A"
    ).otherwise(F.trim(F.col('image_nutrition_url')))
)

In [0]:
#cleaning all the microelements columns

for c in MICROELEMENT_COLS:
    df = df.withColumn(
        c,
        F.when(
            (F.col(c).try_cast('float') < 0) |
            F.col(c).isNull(),
            0
        ).otherwise(F.col(c).try_cast('float'))
    )
    df = df.withColumn(
        c,
        F.round(F.col(c), 2)
    )

In [0]:
#data integrity for energy-kcal_100g
df = df.withColumn(
    'energy-kcal_100g',
    F.col("energy-kj_100g") / 4.184
)
df = df.withColumn(
    'energy-kcal_100g',
    F.round(F.col('energy-kcal_100g'), 2)
)

In [0]:
#data integrity for the fat data columns
df = df.withColumn(
    'saturated-fat_100g',
    F.when(
        F.col('saturated-fat_100g') > F.col("fat_100g"),
        0
    ).otherwise(F.col("saturated-fat_100g"))
)

#Filling null values for remaining cols

In [0]:
cols = [
    "code", "creator", "product_name", "packaging", "packaging_text",
    "brands", "manufacturing_places", "purchase_places", "countries",
    "ingredients_text", "allergens", "owner",
    "completeness"
]

df = df.replace('unknown', 'N/A', subset=cols)
df = df.fillna('N/A', subset=cols)

#Renaming the columns

In [0]:
for old, new in RENAME_MAP.items():
    df = df.withColumnRenamed(old, new)

#Implementing SCD Type 2

In [0]:
COLUMN_LIST = []

for c in COLUMNS_TO_KEEP:
    if c in RENAME_MAP.keys():
        COLUMN_LIST.append(RENAME_MAP[c])
    else:
        COLUMN_LIST.append(c)

In [0]:
source_df = df.withColumn(
    "_hash",
    F.sha2(
        F.concat_ws(
            "||",
            *[
                F.coalesce(
                    F.col(c).cast("string"),
                    F.lit("<NULL>")
                )
                for c in COLUMN_LIST
            ]
        ),
        256
    )
)

target_table = DeltaTable.forName(
    spark,
    SILVER_TABLE_PATH
)

target_df = target_table.toDF()

current_target = target_df.filter(
    F.col("is_current") == True
)

comparison_df = (
    source_df.alias("s")
    .join(
        current_target.alias("t"),
        F.col(f"s.{BUSINESS_KEY}") == F.col(f"t.{BUSINESS_KEY}"),
        "left"
    )
)

In [0]:
new_condition = (
    F.col(f"t.{BUSINESS_KEY}").isNull()
)

changed_condition = (
    F.col(f"t.{BUSINESS_KEY}").isNotNull()
    &
    (
        F.col("s._hash") != F.col("t._hash")
    )
)

new_records = (
    comparison_df
    .filter(new_condition)
    .select(
        *[
            F.col(f"s.{c}").alias(c)
            for c in COLUMN_LIST
        ],
        F.col("s._hash").alias("_hash"),
        F.current_timestamp().alias("change_timestamp")
    )
)

changed_records = (
    comparison_df
    .filter(changed_condition)
    .select(
        *[
            F.col(f"s.{c}").alias(c)
            for c in COLUMN_LIST
        ],
        F.col("s._hash").alias("_hash"),
        F.current_timestamp().alias("change_timestamp")
    )
)

In [0]:
changed_updates = (
    changed_records
    .select(
        *COLUMN_LIST,
        "_hash",
        "change_timestamp"
    )
    .withColumn(
        "merge_key",
        F.col(BUSINESS_KEY)
    )
    .withColumn(
        "action",
        F.lit("UPDATE")
    )
)

changed_inserts = (
    changed_records
    .select(
        *COLUMN_LIST,
        "_hash",
        "change_timestamp"
    )
    .withColumn(
        "merge_key",
        F.lit(None).cast(df.schema[BUSINESS_KEY].dataType)
    )
    .withColumn(
        "action",
        F.lit("INSERT")
    )
)

new_inserts = (
    new_records
    .withColumn(
        "merge_key",
        F.lit(None).cast(df.schema[BUSINESS_KEY].dataType)
    )
    .withColumn(
        "action",
        F.lit("INSERT")
    )
)

staged_source = (
    changed_updates
    .unionByName(changed_inserts)
    .unionByName(new_inserts)
)

In [0]:
(
    target_table.alias("t")
    .merge(
        staged_source.alias("s"),
        f"""
        t.{BUSINESS_KEY} = s.merge_key
        AND t.is_current = true
        """
    )
    .whenMatchedUpdate(
        condition="s.action = 'UPDATE'",
        set={
            "valid_to": "s.change_timestamp",
            "is_current": "false"
        }
    )
    .whenNotMatchedInsert(
        condition="s.action = 'INSERT'",
        values={
            **{
                c:f"s.{c}" for c in COLUMN_LIST
            },
            "_hash": "s._hash",
            "valid_from": "s.change_timestamp",
            "valid_to": "NULL",
            "is_current": "true"
        }
    )
    .execute()
)